# Layer 3 Artifact Construction (Offline)

Build and export frozen artifacts for Layer 3 inference:
- **Ingredient embeddings** (USDA-based: fat/carb/protein ratios, calorie/sodium density, category one-hot)
- **Dish embeddings** (mean ingredients + cooking method + sauce + portion)
- **Similarity neighborhoods** (top-k with macro deltas)
- **Macro delta statistics** (median, IQR, percentiles → refinement bounds)
- **Confidence calibration** (similarity, variance, coverage → scalars/lookup only)

**Output:** `artifacts/` (read-only at inference; no APIs, no training).

In [1]:
# --- Imports & config ---
import json
import os
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv

# Load .env so USDA_API_KEY is available (copy .env.example to .env and add your key)
load_dotenv()

# Config
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(exist_ok=True)

TOP_K_NEIGHBORS = 7
USDA_CSV_PATH = "usda_reference.csv"  # cache path after API fetch
USDA_MAX_INGREDIENTS = 10_000  # target number of ingredients from USDA API
USDA_API_BASE = "https://api.nal.usda.gov/fdc/v1"
USDA_RATE_LIMIT_DELAY_SEC = 4  # stay under 1000 req/hour
INGREDIENT_CATEGORIES = [
    "vegetable", "fruit", "protein", "grain", "dairy", "fat_oil", "legume", "other"
]
COOKING_METHODS_ORDER = [
    "raw", "steamed", "boiled", "baked", "grilled", "fried", "sauteed", "roasted", "other"
]
PORTION_CLASSES = ["small", "medium", "large"]

In [2]:
# --- Input: dishes dataframe (assumed available) ---
# df.columns = dish_id, ingredients (list[str]), cooking_methods (list[str]), sauces,
#              portion_class, calories, fat, carbs, protein, sodium

# If loading from CSV, uncomment and set path:
# df = pd.read_csv("dishes.csv")
# df["ingredients"] = df["ingredients"].apply(eval)  # if stored as string repr of list
# df["cooking_methods"] = df["cooking_methods"].apply(eval)

# For this notebook we create minimal sample data so the pipeline runs end-to-end.
def make_sample_df(n_dishes=50):
    rng = np.random.default_rng(42)
    ingredients_pool = ["chicken breast", "rice", "broccoli", "olive oil", "garlic", "tomato", "onion", "bell pepper", "soy sauce", "butter", "flour", "milk", "egg", "salmon", "spinach", "potato", "carrot", "black beans", "cheese", "lemon"]
    methods_pool = ["baked", "grilled", "steamed", "sauteed", "fried", "roasted", "boiled"]
    portion_pool = ["small", "medium", "large"]
    dishes = []
    for i in range(n_dishes):
        n_ing = rng.integers(3, 8)
        dishes.append({
            "dish_id": f"dish_{i}",
            "ingredients": list(rng.choice(ingredients_pool, size=n_ing, replace=False)),
            "cooking_methods": list(rng.choice(methods_pool, size=rng.integers(1, 3), replace=False)),
            "sauces": rng.uniform(0, 1),
            "portion_class": rng.choice(portion_pool),
            "calories": float(rng.integers(200, 800)),
            "fat": float(rng.integers(5, 50)),
            "carbs": float(rng.integers(20, 100)),
            "protein": float(rng.integers(15, 60)),
            "sodium": float(rng.integers(200, 2000)),
        })
    return pd.DataFrame(dishes)

df = make_sample_df()
df.head(3)

,dish_id,ingredients,cooking_methods,sauces,portion_class,calories,fat,carbs,protein,sodium
0,dish_0,"[salmon, soy sauce, egg]",[fried],0.094177,medium,785.0,38.0,80.0,47.0,1614.0
1,dish_1,"[bell pepper, spinach, soy sauce, butter, broc...",[roasted],0.443414,medium,336.0,9.0,64.0,54.0,314.0
2,dish_2,"[bell pepper, garlic, salmon, broccoli, flour,...","[steamed, grilled]",0.043804,medium,292.0,38.0,74.0,56.0,1540.0


In [3]:
# --- USDA reference: caller to FoodData Central API (10k+ ingredients) ---
# Get a free API key: https://fdc.nal.usda.gov/api-key-signup
# Expected columns: ingredient_name, fat_g, carbs_g, protein_g, calories, sodium_mg, category (per 100g).

import time
import requests

# FDC nutrient IDs (per API docs): 203=Protein, 204=Fat, 205=Carbohydrate, 208=Energy (kcal), 307=Sodium
FDC_NUTRIENT_IDS = {"fat_g": 204, "carbs_g": 205, "protein_g": 203, "calories": 208, "sodium_mg": 307}

# Map FDC foodCategory.description to INGREDIENT_CATEGORIES
def fdc_category_to_ours(desc):
    if not desc:
        return "other"
    d = (desc or "").lower()
    if "vegetable" in d: return "vegetable"
    if "fruit" in d: return "fruit"
    if "meat" in d or "poultry" in d or "fish" in d or "seafood" in d or "egg" in d: return "protein"
    if "cereal" in d or "grain" in d or "bread" in d: return "grain"
    if "dairy" in d or "milk" in d or "cheese" in d: return "dairy"
    if "oil" in d or "fat" in d: return "fat_oil"
    if "legume" in d or "bean" in d or "pea" in d: return "legume"
    return "other"

def _parse_food_nutrients(food):
    """Extract fat_g, carbs_g, protein_g, calories, sodium_mg from foodNutrients (per 100g)."""
    out = {k: 0.0 for k in FDC_NUTRIENT_IDS}
    for n in food.get("foodNutrients") or []:
        num = n.get("nutrientNumber") or n.get("number")
        if num is None:
            continue
        try:
            num = int(num)
        except (TypeError, ValueError):
            continue
        for col, nid in FDC_NUTRIENT_IDS.items():
            if num == nid:
                out[col] = float(n.get("amount") or 0)
                break
    return out

def fetch_usda_via_api(api_key: str, max_ingredients: int = USDA_MAX_INGREDIENTS) -> pd.DataFrame:
    """Call USDA FoodData Central /foods/list (Foundation + SR Legacy) until we have ~max_ingredients."""
    rows = []
    page_size = 200
    page = 1
    url = f"{USDA_API_BASE}/foods/list"
    seen_fdc_ids = set()
    while len(rows) < max_ingredients:
        payload = {
            "dataType": ["Foundation", "SR Legacy"],
            "pageSize": page_size,
            "pageNumber": page,
        }
        params = {"api_key": api_key}
        try:
            r = requests.post(url, json=payload, params=params, timeout=30)
            r.raise_for_status()
            foods = r.json()
        except Exception as e:
            print(f"API error page {page}: {e}")
            break
        if not foods:
            break
        for food in foods:
            fdc_id = food.get("fdcId")
            if fdc_id in seen_fdc_ids:
                continue
            seen_fdc_ids.add(fdc_id)
            name = (food.get("description") or "").strip()
            if not name:
                continue
            nutrients = _parse_food_nutrients(food)
            cat_desc = (food.get("foodCategory") or {})
            if isinstance(cat_desc, dict):
                cat_desc = cat_desc.get("description") or ""
            category = fdc_category_to_ours(cat_desc)
            rows.append({
                "ingredient_name": name,
                "fat_g": nutrients["fat_g"],
                "carbs_g": nutrients["carbs_g"],
                "protein_g": nutrients["protein_g"],
                "calories": nutrients["calories"],
                "sodium_mg": nutrients["sodium_mg"],
                "category": category,
            })
            if len(rows) >= max_ingredients:
                break
        print(f"Page {page}: {len(foods)} foods, total rows {len(rows)}")
        if len(foods) < page_size:
            break
        page += 1
        time.sleep(USDA_RATE_LIMIT_DELAY_SEC)
    return pd.DataFrame(rows)

def load_usda_reference(path=USDA_CSV_PATH):
    if os.path.isfile(path):
        ref = pd.read_csv(path)
        ref = ref.rename(columns={
            "name": "ingredient_name", "Name": "ingredient_name",
            "fat": "fat_g", "Fat": "fat_g", "carbohydrates": "carbs_g", "carbs": "carbs_g",
            "protein": "protein_g", "Protein": "protein_g",
            "calories": "calories", "Calories": "calories", "energy": "calories",
            "sodium": "sodium_mg", "Sodium": "sodium_mg",
        })
        return ref
    return None

# 1) Try cached CSV
usda_df = load_usda_reference()
# 2) If no cache or want more rows, call API (set USDA_API_KEY in env)
if usda_df is None or len(usda_df) < USDA_MAX_INGREDIENTS:
    api_key = os.environ.get("USDA_API_KEY", "").strip()
    if api_key:
        usda_fetched = fetch_usda_via_api(api_key, USDA_MAX_INGREDIENTS)
        if len(usda_fetched) > 0:
            usda_fetched.to_csv(USDA_CSV_PATH, index=False)
            usda_df = usda_fetched
            print(f"Fetched {len(usda_df)} ingredients from USDA API and saved to {USDA_CSV_PATH}")
    elif usda_df is None:
        raise RuntimeError(
            "No USDA cache and USDA_API_KEY not set. "
            "Get a free key at https://fdc.nal.usda.gov/api-key-signup then set env: export USDA_API_KEY=your_key"
        )
    else:
        print(f"Using cached USDA data: {len(usda_df)} ingredients (target was {USDA_MAX_INGREDIENTS})")
if usda_df is not None and "category" not in usda_df.columns:
    usda_df["category"] = "other"
usda_df.head(5)

Page 1: 200 foods, total rows 200
Page 2: 200 foods, total rows 400
Page 3: 200 foods, total rows 600
Page 4: 200 foods, total rows 800
Page 5: 200 foods, total rows 1000
Page 6: 200 foods, total rows 1200
Page 7: 200 foods, total rows 1400
Page 8: 200 foods, total rows 1600
Page 9: 200 foods, total rows 1800
Page 10: 200 foods, total rows 2000
Page 11: 200 foods, total rows 2200
Page 12: 200 foods, total rows 2400
Page 13: 200 foods, total rows 2600
Page 14: 200 foods, total rows 2800
Page 15: 200 foods, total rows 3000
Page 16: 200 foods, total rows 3200
Page 17: 200 foods, total rows 3400
Page 18: 200 foods, total rows 3600
Page 19: 200 foods, total rows 3800
Page 20: 200 foods, total rows 4000
Page 21: 200 foods, total rows 4200
Page 22: 200 foods, total rows 4400
Page 23: 200 foods, total rows 4600
Page 24: 200 foods, total rows 4800
Page 25: 200 foods, total rows 5000
Page 26: 200 foods, total rows 5200
Page 27: 200 foods, total rows 5400
Page 28: 200 foods, total rows 5600
Page 

,ingredient_name,fat_g,carbs_g,protein_g,calories,sodium_mg,category
0,"Abiyuch, raw",0.10,17.60,1.50,69.0,20.0,other
1,"Acerola juice, raw",0.30,4.80,0.40,23.0,3.0,other
2,"Acerola, (west indian cherry), raw",0.30,7.69,0.40,32.0,7.0,other
3,Acorn stew (Apache),3.47,9.22,6.81,95.0,130.0,other
4,"Agave, cooked (Southwest)",0.29,32.00,0.99,135.0,13.0,other


In [4]:
# --- Build ingredient_embeddings ---
# Each embedding: fat_ratio, carb_ratio, protein_ratio, calorie_density, sodium_density, ingredient_category_onehot

usda = usda_df.copy()
usda["total_macro"] = usda["fat_g"] + usda["carbs_g"] + usda["protein_g"]
eps = 1e-6
usda["fat_ratio"] = usda["fat_g"] / (usda["total_macro"] + eps)
usda["carb_ratio"] = usda["carbs_g"] / (usda["total_macro"] + eps)
usda["protein_ratio"] = usda["protein_g"] / (usda["total_macro"] + eps)

# Density: per 100g, then min-max scale to [0,1] across table for comparable dims
cal_max, cal_min = usda["calories"].max(), usda["calories"].min()
usda["calorie_density"] = (usda["calories"] - cal_min) / (cal_max - cal_min + eps)
sd_max, sd_min = usda["sodium_mg"].max(), usda["sodium_mg"].min()
usda["sodium_density"] = (usda["sodium_mg"] - sd_min) / (sd_max - sd_min + eps)

# Category one-hot
cat_to_idx = {c: i for i, c in enumerate(INGREDIENT_CATEGORIES)}
def onehot(cat):
    idx = cat_to_idx.get(cat if isinstance(cat, str) else "other", cat_to_idx["other"])
    arr = np.zeros(len(INGREDIENT_CATEGORIES), dtype=np.float32)
    arr[idx] = 1.0
    return arr
if "category" not in usda.columns:
    usda["category"] = "other"
usda["category_onehot"] = usda["category"].apply(onehot)

ingredient_embeddings = {}
for _, row in usda.iterrows():
    name = row["ingredient_name"].strip().lower()
    emb = np.concatenate([
        np.array([row["fat_ratio"], row["carb_ratio"], row["protein_ratio"],
                  row["calorie_density"], row["sodium_density"]], dtype=np.float32),
        row["category_onehot"],
    ])
    ingredient_embeddings[name] = emb

# For any ingredient in df not in USDA: use mean embedding (so new dishes can be embedded without retraining)
all_ingredients = set()
for ing_list in df["ingredients"]:
    all_ingredients.update(str(x).strip().lower() for x in ing_list)
mean_emb = np.mean(list(ingredient_embeddings.values()), axis=0)
for ing in all_ingredients:
    if ing not in ingredient_embeddings:
        ingredient_embeddings[ing] = mean_emb.copy()

print("Ingredient embedding dim:", len(mean_emb))
print("Num ingredients in dict:", len(ingredient_embeddings))

Ingredient embedding dim: 13
Num ingredients in dict: 8089


In [5]:
# --- Cooking method & portion encodings (for dish embedding) ---
method_to_idx = {m: i for i, m in enumerate(COOKING_METHODS_ORDER)}
portion_to_idx = {p: i for i, p in enumerate(PORTION_CLASSES)}

def encode_cooking_methods(methods):
    vec = np.zeros(len(COOKING_METHODS_ORDER), dtype=np.float32)
    for m in (methods if isinstance(methods, (list, tuple)) else [methods]):
        m = m.strip().lower() if isinstance(m, str) else "other"
        idx = method_to_idx.get(m, method_to_idx["other"])
        vec[idx] = 1.0
    return vec

def encode_portion(pc):
    vec = np.zeros(len(PORTION_CLASSES), dtype=np.float32)
    pc = (pc or "medium").strip().lower()
    idx = portion_to_idx.get(pc, portion_to_idx["medium"])
    vec[idx] = 1.0
    return vec

In [6]:
# --- Build dish_embeddings ---
# dish_embedding = mean(ingredient_embeddings) + cooking_method_vector + sauce_scalar + portion_class_vector

dish_embeddings = {}
for _, row in df.iterrows():
    ings = [str(x).strip().lower() for x in row["ingredients"]]
    mean_ing = np.mean([ingredient_embeddings.get(ing, mean_emb) for ing in ings], axis=0)
    method_vec = encode_cooking_methods(row["cooking_methods"])
    sauce_scalar = np.array([float(row["sauces"])], dtype=np.float32)
    portion_vec = encode_portion(row["portion_class"])
    emb = np.concatenate([mean_ing, method_vec, sauce_scalar, portion_vec])
    dish_embeddings[str(row["dish_id"])] = {
        "embedding": emb,
        "macros": {
            "calories": float(row["calories"]),
            "fat": float(row["fat"]),
            "carbs": float(row["carbs"]),
            "protein": float(row["protein"]),
            "sodium": float(row["sodium"]),
        },
    }

emb_dim = next(iter(dish_embeddings.values()))["embedding"].shape[0]
print("Dish embedding dim:", emb_dim)
print("Dish count:", len(dish_embeddings))

Dish embedding dim: 26
Dish count: 50


In [7]:
# --- Similarity neighborhoods (top-k, cosine; macro_deltas) ---
def cosine_sim(a, b):
    a, b = np.asarray(a, dtype=np.float64), np.asarray(b, dtype=np.float64)
    na = np.linalg.norm(a)
    nb = np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 0.0
    return float(np.dot(a, b) / (na * nb))

dish_ids = list(dish_embeddings.keys())
emb_matrix = np.array([dish_embeddings[did]["embedding"] for did in dish_ids])

neighbors = {}
for i, did in enumerate(dish_ids):
    emb_i = emb_matrix[i]
    sims = []
    for j, other_id in enumerate(dish_ids):
        if j == i:
            continue
        sim = cosine_sim(emb_i, emb_matrix[j])
        macros_i = dish_embeddings[did]["macros"]
        macros_j = dish_embeddings[other_id]["macros"]
        macro_deltas = {}
        for k in macros_i:
            mi, mj = macros_i[k], macros_j[k]
            denom = mi if abs(mi) > 1e-9 else 1.0
            macro_deltas[k] = (mj - mi) / denom
        sims.append((other_id, sim, macro_deltas))
    sims.sort(key=lambda x: -x[1])
    top = sims[:TOP_K_NEIGHBORS]
    neighbors[did] = [
        {"neighbor_id": nid, "similarity": sim, "macro_deltas": mdelta}
        for nid, sim, mdelta in top
    ]

print("Neighbors per dish:", TOP_K_NEIGHBORS)
print("Sample:", list(neighbors.keys())[0], "->", [n["neighbor_id"] for n in neighbors[dish_ids[0]]])

Neighbors per dish: 7
Sample: dish_0 -> ['dish_15', 'dish_26', 'dish_30', 'dish_6', 'dish_1', 'dish_11', 'dish_46']


In [8]:
# --- Macro delta statistics (refinement bounds) ---
# For each macro: median, IQR, variance, 10th & 90th percentiles across all neighbor deltas.

macro_keys = ["calories", "fat", "carbs", "protein", "sodium"]
delta_by_macro = {k: [] for k in macro_keys}
for did, neigh_list in neighbors.items():
    for n in neigh_list:
        for k in macro_keys:
            delta_by_macro[k].append(n["macro_deltas"][k])

macro_delta_stats = {}
for k in macro_keys:
    arr = np.array(delta_by_macro[k])
    macro_delta_stats[k] = {
        "median": float(np.median(arr)),
        "iqr": float(np.percentile(arr, 75) - np.percentile(arr, 25)),
        "variance": float(np.var(arr)),
        "p10": float(np.percentile(arr, 10)),
        "p90": float(np.percentile(arr, 90)),
    }
print("macro_delta_stats (refinement bounds):")
for k, v in macro_delta_stats.items():
    print(f"  {k}: {v}")

macro_delta_stats (refinement bounds):
  calories: {'median': 0.001884463900790377, 'iqr': 0.7307735139052329, 'variance': 0.3341696355928093, 'p10': -0.4941889457955586, 'p90': 0.9770264936213214}
  fat: {'median': -0.020833333333333332, 'iqr': 0.9191028225806451, 'variance': 1.6259186094544118, 'p10': -0.6811870100783874, 'p90': 1.717647058823532}
  carbs: {'median': 0.0, 'iqr': 0.7264679476715714, 'variance': 0.47623569220458256, 'p10': -0.5582277121374866, 'p90': 1.2295454545454552}
  protein: {'median': -0.030462666055886396, 'iqr': 0.710867117117117, 'variance': 0.4243339961624273, 'p10': -0.5424213075060532, 'p90': 1.0}
  sodium: {'median': 0.001967225708078054, 'iqr': 1.0613432105597824, 'variance': 1.6677026818628522, 'p10': -0.6813366254992067, 'p90': 2.0137540453074463}


In [9]:
# --- Confidence calibration stats (scalars & lookup only; no models) ---
# Similarity -> confidence mapping, variance -> penalty, ingredient coverage -> penalty.

# Collect similarity and variance across neighborhoods
sim_list = []
for neigh_list in neighbors.values():
    for n in neigh_list:
        sim_list.append(n["similarity"])
sim_list = np.array(sim_list)

# Similarity -> confidence: simple percentile bins (lookup table)
sim_bins = np.percentile(sim_list, [0, 25, 50, 75, 100])
sim_to_confidence = {
    "bin_edges": sim_bins.tolist(),
    "confidence_at_bin": [0.5, 0.65, 0.8, 0.9, 1.0],  # interpolate in inference
}

# Variance -> penalty: use macro_delta_stats variance per macro
variance_penalty = {k: float(np.clip(np.sqrt(v["variance"]), 0, 2)) for k, v in macro_delta_stats.items()}

# Ingredient coverage: fraction of dish ingredients found in USDA / embedding dict
# Penalty when coverage < 1.0 (e.g. linear drop)
coverage_bins = [0.0, 0.5, 0.75, 1.0]
coverage_penalty = [0.5, 0.2, 0.05, 0.0]

confidence_params = {
    "similarity_to_confidence": sim_to_confidence,
    "variance_penalty": variance_penalty,
    "ingredient_coverage_bins": coverage_bins,
    "ingredient_coverage_penalty": coverage_penalty,
}
print("confidence_params keys:", list(confidence_params.keys()))

confidence_params keys: ['similarity_to_confidence', 'variance_penalty', 'ingredient_coverage_bins', 'ingredient_coverage_penalty']


In [10]:
# --- Export artifacts (read-only at inference) ---
import pickle

ARTIFACTS_DIR.mkdir(exist_ok=True)

with open(ARTIFACTS_DIR / "ingredient_embeddings.pkl", "wb") as f:
    pickle.dump(ingredient_embeddings, f)

with open(ARTIFACTS_DIR / "dish_embeddings.pkl", "wb") as f:
    pickle.dump(dish_embeddings, f)

with open(ARTIFACTS_DIR / "neighbor_index.pkl", "wb") as f:
    pickle.dump(neighbors, f)

with open(ARTIFACTS_DIR / "macro_delta_stats.json", "w") as f:
    json.dump(macro_delta_stats, f, indent=2)

with open(ARTIFACTS_DIR / "confidence_params.json", "w") as f:
    json.dump(confidence_params, f, indent=2)

print("Exported:", list(ARTIFACTS_DIR.iterdir()))

Exported: [PosixPath('artifacts/neighbor_index.pkl'), PosixPath('artifacts/ingredient_embeddings.pkl'), PosixPath('artifacts/confidence_params.json'), PosixPath('artifacts/macro_delta_stats.json'), PosixPath('artifacts/dish_embeddings.pkl')]


In [ ]:
# --- (Optional) Export training data for learned refinement model ---
# Run this to create dishes_training.csv for: python scripts/train_refinement_model.py --data artifacts/dishes_training.csv
df.to_csv(ARTIFACTS_DIR / "dishes_training.csv", index=False)
print("Exported training data:", ARTIFACTS_DIR / "dishes_training.csv", "rows:", len(df))

In [11]:
# --- Sanity: load artifacts in Python ---
with open(ARTIFACTS_DIR / "ingredient_embeddings.pkl", "rb") as f:
    ing_loaded = pickle.load(f)
with open(ARTIFACTS_DIR / "dish_embeddings.pkl", "rb") as f:
    dish_loaded = pickle.load(f)
with open(ARTIFACTS_DIR / "neighbor_index.pkl", "rb") as f:
    neigh_loaded = pickle.load(f)
with open(ARTIFACTS_DIR / "macro_delta_stats.json") as f:
    macro_loaded = json.load(f)
with open(ARTIFACTS_DIR / "confidence_params.json") as f:
    conf_loaded = json.load(f)

assert isinstance(ing_loaded, dict) and all(isinstance(v, np.ndarray) for v in ing_loaded.values())
assert isinstance(dish_loaded, dict)
assert all("embedding" in v and "macros" in v for v in dish_loaded.values())
assert isinstance(neigh_loaded, dict)
assert "fat" in macro_loaded and "median" in macro_loaded["fat"]
assert "similarity_to_confidence" in conf_loaded
print("All artifacts load cleanly.")

All artifacts load cleanly.


**Connection to `layer3/` at runtime:**
- `embeddings.py` loads `ingredient_embeddings.pkl`
- `similarity.py` loads `dish_embeddings.pkl`
- `refinement.py` uses `macro_delta_stats.json` for bounded, explainable refinements
- `confidence.py` uses `confidence_params.json`

New dishes can be embedded using `ingredient_embeddings` (mean for OOV ingredients) without retraining.